# 06 - Embeddings GeoVision-CLIP/SAE para tiles de estaciones

Este notebook toma los tiles Sentinel-2 aceptados del notebook 05 y los pasa por el mismo pipeline de Situación 2:

```text
tile 64x64x12 -> RemoteCLIP 512D -> features auxiliares -> proyector/SAE Situación 2 -> embedding 256D
```


In [1]:
from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from huggingface_hub import hf_hub_download
import open_clip

BASE = Path('/workspace/geovision-cali-hf')
OUT = BASE / 'outputs/situacion3/rubrica/06_embeddings_tiles_estaciones_geovision_clip_sae'
OUT.mkdir(parents=True, exist_ok=True)

PATHS = {
    'station_tiles_metadata': BASE / 'outputs/situacion3/rubrica/05_tiles_estaciones_filtrados_scl/metadata_tiles_estaciones_scl.csv',
    's2_v5b_metadata': BASE / 'outputs/clip_dataset_final_step_by_step/clip_s2_12band_pdf_classes_v5b_high_purity_1500/metadata.jsonl',
    's2_v5b_remoteclip': BASE / 'outputs/clip_training_remoteclip_v10_v5b_embeddings/embeddings_remoteclip_v10_v5b.npz',
    's2_v5b_checkpoint': BASE / 'outputs/clip_training_remoteclip_fusion_ksae_v10_v5b_gsplit_seed_sweep/seed57_k12_drop25_wd1e3_best.pt',
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH = 32
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)

def md5_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

pd.DataFrame([{
    'name': k, 'path': str(v), 'exists': v.exists(),
    'size_mb': round(v.stat().st_size / 1024 / 1024, 3) if v.exists() else None,
    'md5': md5_file(v) if v.exists() and v.is_file() else None,
} for k, v in PATHS.items()])


,name,path,exists,size_mb,md5
0,station_tiles_metadata,/workspace/geovision-cali-hf/outputs/situacion...,True,0.808,b1d97eb3a7b3925125274f58ae090a53
1,s2_v5b_metadata,/workspace/geovision-cali-hf/outputs/clip_data...,True,2.376,5e0a96c7572e00f21edb8e7a864b977d
2,s2_v5b_remoteclip,/workspace/geovision-cali-hf/outputs/clip_trai...,True,2.635,6cb3b11bede28d5568b9253a9a5e1d8d
3,s2_v5b_checkpoint,/workspace/geovision-cali-hf/outputs/clip_trai...,True,9.028,2a512b70e4a837a0c652aa71f2c970b4


In [2]:
meta_all = pd.read_csv(PATHS['station_tiles_metadata'])
meta = meta_all[meta_all['accepted_s2_policy'].fillna(False).astype(bool)].copy().reset_index(drop=True)
meta['pair_id'] = meta['tile_id'].astype(str)
meta['date_day'] = pd.to_datetime(meta['date_day']).dt.strftime('%Y-%m-%d')

s2_meta = pd.read_json(PATHS['s2_v5b_metadata'], lines=True)
s2_npz = np.load(PATHS['s2_v5b_remoteclip'], allow_pickle=True)
s2_remote = s2_npz['remoteclip_visual_512'].astype('float32')

summary_load = {
    'accepted_station_tiles': int(len(meta)),
    'stations': int(meta['estacion'].nunique()),
    'grid_cells': int(meta['grid_id'].nunique()),
    'dates': int(meta['date_day'].nunique()),
    's2_training_tiles': int(len(s2_meta)),
    's2_remote_shape': tuple(s2_remote.shape),
}
summary_load


{'accepted_station_tiles': 613,
 'stations': 9,
 'grid_cells': 9,
 'dates': 107,
 's2_training_tiles': 1500,
 's2_remote_shape': (1500, 512)}

In [3]:
def s2_to_rgb(path):
    arr = np.load(path).astype('float32')
    rgb = arr[:, :, [3, 2, 1]]
    lo = np.nanpercentile(rgb, 2, axis=(0, 1), keepdims=True)
    hi = np.nanpercentile(rgb, 98, axis=(0, 1), keepdims=True)
    rgb = np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1).astype('float32')
    ten = torch.from_numpy(rgb).permute(2, 0, 1)
    ten = F.interpolate(ten.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False).squeeze(0)
    return (ten - CLIP_MEAN) / CLIP_STD

remote_ckpt = hf_hub_download('chendelong/RemoteCLIP', 'RemoteCLIP-ViT-B-32.pt', cache_dir=str(BASE / 'checkpoints' / 'remoteclip'))
clip_model, _, _ = open_clip.create_model_and_transforms('ViT-B-32')
clip_state = torch.load(remote_ckpt, map_location='cpu')
clip_model.load_state_dict(clip_state, strict=True)
clip_model = clip_model.to(DEVICE).eval()
for p in clip_model.parameters():
    p.requires_grad = False

remote_embs = []
paths = meta['image_path'].astype(str).tolist()
with torch.no_grad():
    for start in range(0, len(paths), BATCH):
        batch_paths = paths[start:start + BATCH]
        imgs = torch.stack([s2_to_rgb(p) for p in batch_paths]).to(DEVICE)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            z = clip_model.encode_image(imgs).float()
        z = F.normalize(z, dim=-1)
        remote_embs.append(z.cpu().numpy().astype('float32'))
        print(json.dumps({'stage': 'remoteclip_embedding', 'done': min(start + len(batch_paths), len(paths)), 'total': len(paths)}, ensure_ascii=False), flush=True)

remote_station = np.concatenate(remote_embs, axis=0).astype('float32')
{'remote_station_shape': tuple(remote_station.shape), 'nan_count': int(np.isnan(remote_station).sum())}


/tmp/ipykernel_973558/2759538728.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clip_state = torch.load(remote_ckpt, map_location='cpu')


/tmp/ipykernel_973558/2759538728.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):


{"stage": "remoteclip_embedding", "done": 32, "total": 613}


{"stage": "remoteclip_embedding", "done": 64, "total": 613}


{"stage": "remoteclip_embedding", "done": 96, "total": 613}


...

{"stage": "remoteclip_embedding", "done": 544, "total": 613}


{"stage": "remoteclip_embedding", "done": 576, "total": 613}


{"stage": "remoteclip_embedding", "done": 608, "total": 613}


{"stage": "remoteclip_embedding", "done": 613, "total": 613}


{'remote_station_shape': (613, 512), 'nan_count': 0}

In [4]:
def aux_row_from_image_and_meta(r):
    img = np.load(r.image_path).astype('float32')
    red = img[:, :, 3]
    green = img[:, :, 2]
    nir = img[:, :, 7]
    swir = img[:, :, 10]
    ndvi = (nir - red) / (nir + red + 1e-6)
    ndbi = (swir - nir) / (swir + nir + 1e-6)
    ndwi = (green - nir) / (green + nir + 1e-6)
    arr = img.reshape(-1, 12)
    vals = []
    vals.extend(arr.mean(0))
    vals.extend(arr.std(0))
    vals.extend(np.percentile(arr, [10, 50, 90], axis=0).ravel())
    for idx in [ndvi, ndbi, ndwi]:
        vals.extend([np.nanmean(idx), np.nanstd(idx), np.nanpercentile(idx, 10), np.nanpercentile(idx, 50), np.nanpercentile(idx, 90)])
    d = pd.to_datetime(r.date_day)
    doy = d.dayofyear
    vals.extend([float(d.year), np.sin(2*np.pi*doy/366), np.cos(2*np.pi*doy/366), float(r.ndvi_mean), float(r.ndbi_mean), float(r.ndwi_mean), float(r.scl_cloud_shadow_pct), float(r.scl_valid_visual_pct)])
    return np.array(vals, dtype='float32')

aux_station = np.stack([aux_row_from_image_and_meta(r) for _, r in meta.iterrows()]).astype('float32')

def aux_row_s2(r):
    return aux_row_from_image_and_meta(r)
aux_s2 = np.stack([aux_row_s2(r) for _, r in s2_meta.iterrows()]).astype('float32')
train_mask = s2_meta['split'].to_numpy() == 'train'
sc_remote = StandardScaler().fit(s2_remote[train_mask])
sc_aux = StandardScaler().fit(aux_s2[train_mask])
X_595_station = np.concatenate([sc_remote.transform(remote_station), sc_aux.transform(aux_station)], axis=1).astype('float32')
{'aux_station_shape': tuple(aux_station.shape), 'X_595_station_shape': tuple(X_595_station.shape), 'nan_count': int(np.isnan(X_595_station).sum())}


{'aux_station_shape': (613, 83),
 'X_595_station_shape': (613, 595),
 'nan_count': 0}

In [5]:
ckpt = torch.load(PATHS['s2_v5b_checkpoint'], map_location='cpu')
IN = 595
EMB = 256
SAE_H = 1024
k_frac = ckpt['config']['k_frac']
drop = ckpt['config']['drop']

class SAE(nn.Module):
    def __init__(self, k_frac=0.15):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(EMB, SAE_H), nn.ReLU())
        self.dec = nn.Linear(SAE_H, EMB)
        self.k_frac = k_frac
    def forward(self, x):
        z = self.enc(x)
        k = max(1, int(z.shape[1] * self.k_frac))
        vals, idx = torch.topk(z, k, dim=1)
        sparse = torch.zeros_like(z).scatter(1, idx, vals)
        return sparse, self.dec(sparse)

class VisualProjectorSAE(nn.Module):
    def __init__(self, drop=0.25, k_frac=0.12):
        super().__init__()
        self.img = nn.Sequential(nn.LayerNorm(IN), nn.Linear(IN, 768), nn.GELU(), nn.Dropout(drop), nn.Linear(768, 512), nn.GELU(), nn.Dropout(drop), nn.Linear(512, EMB))
        self.sae_i = SAE(k_frac)
    def forward_img(self, x):
        h = self.img(x)
        z, r = self.sae_i(h)
        return h, z, r

model = VisualProjectorSAE(drop=drop, k_frac=k_frac).to(DEVICE)
visual_state = {k: v for k, v in ckpt['model_state_dict'].items() if k.startswith('img.') or k.startswith('sae_i.')}
missing, unexpected = model.load_state_dict(visual_state, strict=False)
model.eval()
with torch.no_grad():
    x_t = torch.tensor(X_595_station, dtype=torch.float32, device=DEVICE)
    h_256, z_1024, r_256 = model.forward_img(x_t)
h_256 = h_256.cpu().numpy().astype('float32')
z_1024 = z_1024.cpu().numpy().astype('float32')
r_256 = r_256.cpu().numpy().astype('float32')
{'missing': missing, 'unexpected': unexpected, 'h_256_shape': tuple(h_256.shape), 'r_256_shape': tuple(r_256.shape), 'z_1024_sparsity': float((np.abs(z_1024) < 1e-8).mean())}


/tmp/ipykernel_973558/2630163215.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(PATHS['s2_v5b_checkpoint'], map_location='cpu')


{'missing': [],
 'unexpected': [],
 'h_256_shape': (613, 256),
 'r_256_shape': (613, 256),
 'z_1024_sparsity': 0.880859375}

In [6]:
out_npz = OUT / 'embeddings_estaciones_geovision_clip_sae_256.npz'
meta_out = OUT / 'metadata_embeddings_estaciones.csv'
summary_out = OUT / 'summary_embeddings_estaciones.json'
manifest_out = OUT / 'manifest_06_embeddings_tiles_estaciones_geovision_clip_sae.json'

np.savez_compressed(out_npz, remoteclip_visual_512=remote_station, h_256=h_256, r_256=r_256, z_1024=z_1024, pair_id=meta['pair_id'].astype(str).to_numpy(), image_path=meta['image_path'].astype(str).to_numpy(), grid_id=meta['grid_id'].astype(str).to_numpy(), date_day=meta['date_day'].astype(str).to_numpy())
meta.to_csv(meta_out, index=False)
summary = {
    'n_tiles': int(len(meta)),
    'remoteclip_visual_512_shape': list(remote_station.shape),
    'h_256_shape': list(h_256.shape),
    'r_256_shape': list(r_256.shape),
    'z_1024_shape': list(z_1024.shape),
    'z_1024_sparsity_ratio': float((np.abs(z_1024) < 1e-8).mean()),
    'n_stations': int(meta['estacion'].nunique()),
    'n_grid_cells': int(meta['grid_id'].nunique()),
    'n_dates': int(meta['date_day'].nunique()),
    'date_min': str(pd.to_datetime(meta['date_day']).min().date()),
    'date_max': str(pd.to_datetime(meta['date_day']).max().date()),
}
summary_out.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
manifest = {'notebook': '06_embeddings_tiles_estaciones_geovision_clip_sae.ipynb', 'inputs': {k: str(v) for k, v in PATHS.items()}, 'input_md5': {k: md5_file(v) for k, v in PATHS.items() if v.exists() and v.is_file()}, 'outputs': {'embeddings_npz': str(out_npz), 'metadata': str(meta_out), 'summary': str(summary_out)}, 'summary': summary}
manifest_out.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
summary


{'n_tiles': 613,
 'remoteclip_visual_512_shape': [613, 512],
 'h_256_shape': [613, 256],
 'r_256_shape': [613, 256],
 'z_1024_shape': [613, 1024],
 'z_1024_sparsity_ratio': 0.880859375,
 'n_stations': 9,
 'n_grid_cells': 9,
 'n_dates': 107,
 'date_min': '2020-01-02',
 'date_max': '2024-12-16'}